# Project 2: Student Academic Performance Analytics
**Industry:** Education / EdTech  
**Dataset:** `02_student_performance.csv`

### Task 1: Load and Inspect the Dataset

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('02_student_performance.csv')

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("First 5 Rows:")
display(df.head())

print("\nDataset Info:")
df.info()

print("\nSummary Statistics:")
display(df.describe(include='all'))

### Task 2: Report Missing Values and Duplicate Student_ID Values

In [ ]:
# Check missing values per column
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found.")
print("\nFull Missing Value Breakdown:")
print(missing_values)

# Check duplicate Student_ID values
duplicate_ids = df['Student_ID'].duplicated().sum()
print(f"\nDuplicate Student_ID count: {duplicate_ids}")

### Task 3: Handle Missing Attendance Values and Explain the Chosen Method

In [ ]:
# Compare mean vs median before imputation
mean_att = df['Attendance_Percent'].mean()
median_att = df['Attendance_Percent'].median()
print(f"Mean Attendance: {mean_att:.2f}% | Median Attendance: {median_att:.2f}%")

# Impute missing values using the median
df['Attendance_Percent'] = df['Attendance_Percent'].fillna(median_att)

print(f"Missing Attendance values after imputation: {df['Attendance_Percent'].isnull().sum()}")

**Explanation of Method:**  
Median imputation (`78.2%`) was chosen over mean imputation (`78.15%`) or dropping rows. Because attendance percentages are bounded at 100% and can have left-skewed low outliers (minimum is `41.2%`), the median is more robust against skewness while preserving all 500 student records for downstream subject and department analyses.

### Task 4: Check Score Columns for Values Outside 0–100

In [ ]:
score_cols = ['Python_Marks', 'Maths_Marks', 'Physics_Marks']

print("Score Validation Check (Valid Range: 0 to 100):")
for col in score_cols:
    out_of_range = df[(df[col] < 0) | (df[col] > 100)]
    print(f"- {col}: {len(out_of_range)} invalid records (Min: {df[col].min()}, Max: {df[col].max()})")

### Task 5: Calculate Total and Average Marks for Each Student

In [ ]:
# Calculate Total and Average marks across the 3 subjects
df['Calculated_Total'] = df[['Python_Marks', 'Maths_Marks', 'Physics_Marks']].sum(axis=1)
df['Average_Marks'] = (df['Calculated_Total'] / 3).round(2)

# Verify alignment with existing Total_Marks column
discrepancies = (df['Total_Marks'] != df['Calculated_Total']).sum()
print(f"Discrepancies with existing Total_Marks column: {discrepancies}")

display(df[['Student_ID', 'Python_Marks', 'Maths_Marks', 'Physics_Marks', 'Total_Marks', 'Average_Marks']].head())

### Task 6: Create a Grade Column Using Clearly Defined Score Ranges

In [ ]:
# Define grade boundaries: F (<40), D (40–59.99), C (60–74.99), B (75–89.99), A (90–100)
grade_bins = [0, 40, 60, 75, 90, 101]
grade_labels = ['F', 'D', 'C', 'B', 'A']

df['Grade'] = pd.cut(df['Average_Marks'], bins=grade_bins, labels=grade_labels, right=False)

grade_dist = df['Grade'].value_counts().sort_index().reset_index()
grade_dist.columns = ['Grade', 'Student_Count']
grade_dist['Percentage'] = (grade_dist['Student_Count'] / len(df) * 100).round(2)
display(grade_dist)

### Task 7: Calculate Department-Wise Average Marks

In [ ]:
dept_summary = df.groupby('Department').agg(
    Student_Count=('Student_ID', 'count'),
    Avg_Python=('Python_Marks', 'mean'),
    Avg_Maths=('Maths_Marks', 'mean'),
    Avg_Physics=('Physics_Marks', 'mean'),
    Avg_Total_Marks=('Total_Marks', 'mean'),
    Overall_Avg_Marks=('Average_Marks', 'mean'),
    Avg_Attendance=('Attendance_Percent', 'mean')
).round(2).sort_values(by='Overall_Avg_Marks', ascending=False)

display(dept_summary)

### Task 8: Find the Top 10 Students by Average Marks

In [ ]:
top_10_students = df.sort_values(by=['Average_Marks', 'Total_Marks'], ascending=False).head(10)

display(top_10_students[['Student_ID', 'Department', 'Python_Marks', 'Maths_Marks', 'Physics_Marks', 'Total_Marks', 'Average_Marks', 'Grade']])

### Task 9: Identify Students Below 40% Average Marks

In [ ]:
# Check students with overall Average_Marks < 40%
below_40 = df[df['Average_Marks'] < 40]
print(f"Students with overall Average_Marks < 40%: {len(below_40)}")
print(f"Lowest overall Average_Marks in dataset: {df['Average_Marks'].min()}%")

# Additional check: Students flagged as 'Fail' (individual subject score < 40)
subject_fail = df[df['Result'] == 'Fail']
print(f"\nStudents with Result == 'Fail' (scored <40 in at least one subject): {len(subject_fail)}")
display(subject_fail[['Student_ID', 'Department', 'Python_Marks', 'Maths_Marks', 'Physics_Marks', 'Average_Marks', 'Result']].head(10))

### Task 10: Compare Average Marks for Different Attendance Bands

In [ ]:
att_bins = [0, 60, 75, 85, 101]
att_labels = ['Low (<60%)', 'Moderate (60-74.9%)', 'Good (75-84.9%)', 'Excellent (85-100%)']
df['Attendance_Band'] = pd.cut(df['Attendance_Percent'], bins=att_bins, labels=att_labels, right=False)

attendance_analysis = df.groupby('Attendance_Band', observed=False).agg(
    Student_Count=('Student_ID', 'count'),
    Mean_Avg_Marks=('Average_Marks', 'mean'),
    Median_Avg_Marks=('Average_Marks', 'median'),
    Pass_Rate_Pct=('Result', lambda x: (x == 'Pass').mean() * 100)
).round(2)

display(attendance_analysis)

### Task 11: Analyze the Relationship Between Study Hours and Average Marks

In [ ]:
# Pearson correlation
corr_study = df['Study_Hours_Per_Day'].corr(df['Average_Marks'])
print(f"Pearson Correlation (Study_Hours_Per_Day vs Average_Marks): {corr_study:.4f}\n")

# Compare study habit tiers using np.where()
df['Study_Group'] = np.where(df['Study_Hours_Per_Day'] >= 3.0, '3+ Hours/Day', '<3 Hours/Day')

study_summary = df.groupby('Study_Group').agg(
    Student_Count=('Student_ID', 'count'),
    Mean_Study_Hours=('Study_Hours_Per_Day', 'mean'),
    Mean_Avg_Marks=('Average_Marks', 'mean'),
    Median_Avg_Marks=('Average_Marks', 'median')
).round(2)

display(study_summary)

### Task 12: Compare Results for Students With and Without Internet Access

In [ ]:
internet_comparison = df.groupby('Internet_Access').agg(
    Student_Count=('Student_ID', 'count'),
    Mean_Avg_Marks=('Average_Marks', 'mean'),
    Median_Avg_Marks=('Average_Marks', 'median'),
    Mean_Python=('Python_Marks', 'mean'),
    Mean_Assignments=('Assignments_Submitted', 'mean'),
    Pass_Rate_Pct=('Result', lambda x: (x == 'Pass').mean() * 100)
).round(2)

display(internet_comparison)

### Task 13: Calculate Median, Standard Deviation, and Quartiles of Python_Marks Using NumPy

In [ ]:
python_arr = df['Python_Marks'].to_numpy()

py_median = np.median(python_arr)
py_std_sample = np.std(python_arr, ddof=1)  # Sample standard deviation
py_std_pop = np.std(python_arr, ddof=0)     # Population standard deviation
q1, q2, q3 = np.percentile(python_arr, [25, 50, 75])

print(f"Python_Marks Statistics (NumPy):")
print(f"- Q1 (25th Percentile): {q1:.2f}")
print(f"- Median / Q2 (50th Percentile): {py_median:.2f}")
print(f"- Q3 (75th Percentile): {q3:.2f}")
print(f"- Interquartile Range (IQR): {q3 - q1:.2f}")
print(f"- Standard Deviation (Sample, ddof=1): {py_std_sample:.2f}")
print(f"- Standard Deviation (Population, ddof=0): {py_std_pop:.2f}")

### Task 14: Identify Potential Score Outliers Using IQR

In [ ]:
cols_to_check = ['Python_Marks', 'Maths_Marks', 'Physics_Marks', 'Average_Marks']

for col in cols_to_check:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"=== {col} ===")
    print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f} | Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"Outliers Found: {len(outliers)}")
    if not outliers.empty:
        display(outliers[['Student_ID', 'Department', col, 'Result']])
    print()

### Task 15: Five Recommendations for Academic Support

1. **Target Single-Subject Remediation for the 18 Failing Students:** While zero students fell below a 40% overall average, 18 students failed due to scoring under 40 in a single subject—most notably in Physics (4 IQR outliers below 31.5) and Maths (lowest overall subject mean at 67.21). Implement subject-specific remedial workshops focused on Physics and Maths problem-solving for students scoring below 45 in any single assessment.
2. **Introduce Early-Warning Attendance Nudges Below 60%:** Students with attendance below 60% (43 students) recorded the lowest mean average marks (67.81% vs. 70.44% for the 75–84.9% band). Configure automated LMS alerts to trigger advisor check-ins whenever a student's attendance drops below 65%.
3. **Provide Offline Study Kits & Lab Access for Students Without Internet:** Although students without home internet (58 students) maintain comparable average marks (69.63%), their pass rate drops by 5.68 percentage points (91.38% vs. 97.06%). Providing downloadable Python/coding environments, offline PDF modules, and extended campus computer lab hours will close this pass-rate gap.
4. **Shift Focus from Study Duration to Guided Study Techniques:** Because daily study hours show near-zero correlation with average marks ($r = 0.035$), simply telling struggling students to study longer is ineffective. Replace generic study-hour targets with structured peer-mentoring sessions and active-recall practice tests led by top-performing Grade A/B students.
5. **Prioritize Immediate Counseling for Multi-Risk Outliers (`STU0144` and `STU0395`):** Students flagged as low outliers in overall `Average_Marks` (`STU0144` at 43.67% and `STU0395` at 44.00%) require individualized academic recovery plans combining faculty mentorship, assignment completion tracking, and foundational tutoring before final exams.